# The Current State of Solar Modeling — Implementation\n# 태양 모델링의 현재 상태 — 구현\n\n**Paper**: Christensen-Dalsgaard et al. (1996), *Science*, 272, 1286–1292\n\nThis notebook implements key concepts from the paper:\n이 노트북은 논문의 핵심 개념을 구현합니다:\n\n1. **Standard Solar Model (SSM) structure** — hydrostatic equilibrium & polytropic model\n   표준 태양 모델 구조 — 정역학적 평형 & 폴리트로프 모델\n2. **Sound speed profile** — comparing ideal gas vs. realistic profiles\n   음속 프로파일 — 이상 기체 vs. 현실적 프로파일 비교\n3. **p-mode turning points** — computing inner turning points for different (ν, ℓ)\n   p-mode 전환점 — 서로 다른 (ν, ℓ)에 대한 내부 전환점 계산\n4. **Gamow peak** — nuclear reaction rate window in the solar core\n   Gamow 피크 — 태양 핵에서의 핵반응률 에너지 창\n5. **EOS comparison** — effect of different equations of state on γ₁\n   상태방정식 비교 — 서로 다른 EOS가 γ₁에 미치는 효과

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.integrate import solve_ivp\nfrom scipy.optimize import brentq\n\nplt.rcParams.update({\n    \"figure.figsize\": (10, 6),\n    \"font.size\": 12,\n    \"axes.grid\": True,\n    \"grid.alpha\": 0.3,\n})\n\n# Physical constants\nG = 6.674e-8        # gravitational constant [cm^3 g^-1 s^-2]\nk_B = 1.381e-16     # Boltzmann constant [erg K^-1]\nm_p = 1.673e-24     # proton mass [g]\na_rad = 7.566e-15   # radiation constant [erg cm^-3 K^-4]\nc_light = 3e10      # speed of light [cm s^-1]\nsigma_sb = 5.67e-5  # Stefan-Boltzmann [erg cm^-2 s^-2 K^-4]\n\n# Solar parameters from the paper\nM_sun = 1.989e33    # solar mass [g]\nR_sun = 6.96e10     # solar radius [cm]\nL_sun = 3.846e33    # solar luminosity [erg s^-1]\nT_c = 1.57e7        # central temperature [K]\nrho_c = 153.0       # central density [g cm^-3]\nage_sun = 4.52e9    # solar age [years]\n\nprint(\"Constants and solar parameters loaded.\")\nprint(f\"Solar mass:   {M_sun:.3e} g\")\nprint(f\"Solar radius: {R_sun:.3e} cm\")\nprint(f\"Solar luminosity: {L_sun:.3e} erg/s\")\nprint(f\"Central temperature: {T_c:.2e} K\")\nprint(f\"Central density: {rho_c:.1f} g/cm³\")

## 1. Polytropic Solar Model / 폴리트로프 태양 모델\n\nThe Lane-Emden equation describes a self-gravitating polytropic sphere in hydrostatic equilibrium. For a polytropic index $n$, the pressure-density relation is $P = K\\rho^{1+1/n}$, and the structure is governed by:\n\nLane-Emden 방정식은 정역학적 평형 상태의 자기중력 폴리트로프 구를 기술합니다:\n\n$$\\frac{1}{\\xi^2}\\frac{d}{d\\xi}\\left(\\xi^2 \\frac{d\\theta}{d\\xi}\\right) = -\\theta^n$$\n\nwhere $\\xi$ is dimensionless radius and $\\theta$ is dimensionless density ($\\rho = \\rho_c \\theta^n$).\n\nFor the Sun, $n = 3$ (Eddington's standard model) is a reasonable approximation for the radiative interior.

In [ ]:
def lane_emden(xi, y, n):\n    \"\"\"Lane-Emden equation: d/dxi(xi^2 dtheta/dxi) = -xi^2 theta^n.\n\n    Args:\n        xi: Dimensionless radius.\n        y: [theta, dtheta/dxi].\n        n: Polytropic index.\n\n    Returns:\n        Derivatives [dtheta/dxi, d2theta/dxi2].\n    \"\"\"\n    theta, dtheta = y\n    if theta < 0:\n        theta = 0.0\n    d2theta = -theta**n - 2.0 / xi * dtheta if xi > 0 else -theta**n / 3.0\n    return [dtheta, d2theta]\n\n\ndef solve_lane_emden(n, xi_max=10.0, n_points=10000):\n    \"\"\"Solve the Lane-Emden equation for polytropic index n.\n\n    Args:\n        n: Polytropic index.\n        xi_max: Maximum dimensionless radius.\n        n_points: Number of output points.\n\n    Returns:\n        xi, theta, dtheta arrays.\n    \"\"\"\n    xi_span = (1e-6, xi_max)\n    xi_eval = np.linspace(1e-6, xi_max, n_points)\n\n    # Boundary conditions: theta(0) = 1, dtheta/dxi(0) = 0\n    # Start from small xi with series expansion:\n    # theta ≈ 1 - xi^2/6 + n*xi^4/120\n    xi0 = 1e-6\n    theta0 = 1 - xi0**2 / 6 + n * xi0**4 / 120\n    dtheta0 = -xi0 / 3 + n * xi0**3 / 30\n\n    def event_zero(xi, y, n):\n        return y[0]\n    event_zero.terminal = True\n    event_zero.direction = -1\n\n    sol = solve_ivp(\n        lane_emden, xi_span, [theta0, dtheta0], args=(n,),\n        t_eval=xi_eval, events=event_zero, dense_output=True,\n        rtol=1e-10, atol=1e-12,\n    )\n\n    # Trim to where theta >= 0\n    mask = sol.y[0] >= 0\n    return sol.t[mask], sol.y[0][mask], sol.y[1][mask]\n\n\n# Solve for n = 3 (Eddington standard model)\nxi3, theta3, dtheta3 = solve_lane_emden(n=3)\nxi1_surface = xi3[-1]\nprint(f\"n=3 polytrope: surface at ξ₁ = {xi1_surface:.4f} (analytic: 6.8969)\")\nprint(f\"  -ξ₁ · θ'(ξ₁) = {-xi1_surface * dtheta3[-1]:.4f} (analytic: 2.0182)\")\n\n# Also solve n=1.5 (convective) and n=3 for comparison\nxi15, theta15, dtheta15 = solve_lane_emden(n=1.5)\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Left: theta vs xi\nax = axes[0]\nax.plot(xi3 / xi3[-1], theta3, \"b-\", lw=2, label=\"n=3 (radiative)\")\nax.plot(xi15 / xi15[-1], theta15, \"r--\", lw=2, label=\"n=1.5 (convective)\")\nax.set_xlabel(r\"$r/R$\")\nax.set_ylabel(r\"$\\theta$ (dimensionless temperature)\")\nax.set_title(\"Lane-Emden Solutions / Lane-Emden 해\")\nax.legend()\nax.set_xlim(0, 1)\nax.set_ylim(0, 1.05)\n\n# Right: density profile rho/rho_c = theta^n\nax = axes[1]\nrho_n3 = theta3**3\nrho_n15 = theta15**1.5\nax.plot(xi3 / xi3[-1], rho_n3, \"b-\", lw=2, label=r\"n=3: $\\rho/\\rho_c = \\theta^3$\")\nax.plot(xi15 / xi15[-1], rho_n15, \"r--\", lw=2, label=r\"n=1.5: $\\rho/\\rho_c = \\theta^{1.5}$\")\nax.set_xlabel(r\"$r/R$\")\nax.set_ylabel(r\"$\\rho / \\rho_c$\")\nax.set_title(\"Density Profile / 밀도 프로파일\")\nax.legend()\nax.set_xlim(0, 1)\nax.set_ylim(0, 1.05)\n\nplt.tight_layout()\nplt.show()

## 2. Sound Speed Profile / 음속 프로파일\n\nThe adiabatic sound speed is $c^2 = \\gamma_1 P / \\rho$. For a fully ionized ideal gas:\n단열 음속: $c^2 = \\gamma_1 P / \\rho$. 완전 이온화 이상 기체에서:\n\n$$c^2 = \\frac{\\gamma_1 k_B T}{\\mu m_p}$$\n\nwhere $\\mu$ is mean molecular weight. We construct a simplified sound-speed profile using the $n=3$ polytrope and compare with Model S values from the paper ($\\delta c^2/c^2 < 0.5\\%$).

In [ ]:
# Build sound speed profile from n=3 polytrope\n# For n=3 polytrope: P = K*rho^(4/3), T ∝ theta\n# gamma_1 = 5/3 for fully ionized ideal gas\n\ngamma_1 = 5.0 / 3.0\nmu_core = 0.62   # mean molecular weight (fully ionized, X=0.34, Y=0.64)\nmu_surface = 0.60  # slightly different due to composition gradient\n\n# Temperature profile: T = T_c * theta (for n=3 polytrope)\nr_frac = xi3 / xi3[-1]  # r/R_sun\nT_profile = T_c * theta3\n\n# Sound speed: c^2 = gamma_1 * k_B * T / (mu * m_p)\n# Use a simple mu profile (linearly interpolated)\nmu_profile = mu_core + (mu_surface - mu_core) * r_frac\nc_sq = gamma_1 * k_B * T_profile / (mu_profile * m_p)\nc_profile = np.sqrt(c_sq) / 1e5  # convert to km/s\n\n# Model S approximate values (from Bahcall et al. and helioseismic inversions)\n# Central sound speed ~500 km/s, surface ~10 km/s\n# CZ base at r/R ≈ 0.713: c ≈ 220 km/s\nprint(f\"Central sound speed: {c_profile[0]:.1f} km/s (Model S: ~506 km/s)\")\nprint(f\"Sound speed at r/R=0.71: ~{np.interp(0.71, r_frac, c_profile):.1f} km/s\")\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Left: sound speed profile\nax = axes[0]\nax.plot(r_frac, c_profile, \"b-\", lw=2, label=\"n=3 polytrope\")\nax.axvline(x=0.713, color=\"red\", ls=\"--\", alpha=0.7, label=r\"CZ base ($r/R=0.713$)\")\nax.set_xlabel(r\"$r / R_\\odot$\")\nax.set_ylabel(\"Sound speed [km/s] / 음속\")\nax.set_title(\"Sound Speed Profile / 음속 프로파일\")\nax.legend()\nax.set_xlim(0, 1)\n\n# Right: c/r profile (relevant for turning points)\nc_over_r = c_profile / (r_frac + 1e-10)  # avoid division by zero\nax = axes[1]\nvalid = r_frac > 0.01\nax.plot(r_frac[valid], c_over_r[valid], \"b-\", lw=2)\nax.axvline(x=0.713, color=\"red\", ls=\"--\", alpha=0.7, label=\"CZ base\")\nax.set_xlabel(r\"$r / R_\\odot$\")\nax.set_ylabel(r\"$c(r)/r$ [km/s per $R_\\odot$]\")\nax.set_title(r\"$c/r$ Profile (determines turning points) / 전환점 결정 프로파일\")\nax.legend()\nax.set_xlim(0, 1)\nax.set_ylim(0, np.max(c_over_r[valid]) * 1.1)\n\nplt.tight_layout()\nplt.show()

## 3. p-Mode Turning Points / p-Mode 전환점\n\nFrom the paper (Eq. in Fig. 1 caption), the inner turning point $r_t$ of a p-mode is determined by:\n논문의 Fig. 1 캡션에 따르면, p-mode의 내부 전환점 $r_t$는 다음으로 결정됩니다:\n\n$$\\frac{c(r_t)}{r_t} = \\frac{2\\pi\\nu}{\\ell + 1/2}$$\n\nLow-$\\ell$ modes penetrate to the core; high-$\\ell$ modes are trapped near the surface.\n낮은 $\\ell$ 모드는 핵까지 침투하고, 높은 $\\ell$ 모드는 표면 근처에 갇힙니다.\n\nThis is the basis for helioseismic \"imaging\" — by observing modes of different $\\ell$, we probe different depths.\n이것이 일진학적 \"영상화\"의 기초입니다 — 서로 다른 $\\ell$의 모드를 관측하여 서로 다른 깊이를 탐사합니다.

In [ ]:
def find_turning_point(r_frac, c_km, nu_uhz, ell):\n    \"\"\"Find inner turning point for a p-mode.\n\n    The turning point satisfies c(r_t)/r_t = 2*pi*nu/(ell + 0.5).\n    Here r is in units of R_sun, c in km/s.\n\n    Args:\n        r_frac: r/R_sun array.\n        c_km: Sound speed in km/s.\n        nu_uhz: Frequency in microhertz.\n        ell: Spherical harmonic degree.\n\n    Returns:\n        Turning point r_t/R_sun, or NaN if not found.\n    \"\"\"\n    nu_hz = nu_uhz * 1e-6\n    # c/r in physical units: c(km/s) / (r * R_sun_km)\n    R_sun_km = R_sun / 1e5\n    target = 2 * np.pi * nu_hz / (ell + 0.5)  # in s^-1\n    # c(r)/r in s^-1: c_km * 1e5 / (r_frac * R_sun)\n    c_over_r_phys = c_km * 1e5 / (r_frac * R_sun)  # in s^-1\n\n    # Find where c/r = target (scanning from center outward)\n    valid = r_frac > 0.005\n    r_v = r_frac[valid]\n    cor_v = c_over_r_phys[valid]\n\n    # c/r generally decreases outward; turning point is where c/r crosses target\n    diff = cor_v - target\n    crossings = np.where(np.diff(np.sign(diff)))[0]\n    if len(crossings) == 0:\n        return np.nan\n    # Take the innermost crossing\n    idx = crossings[0]\n    # Linear interpolation\n    r_t = r_v[idx] + (r_v[idx + 1] - r_v[idx]) * (-diff[idx]) / (diff[idx + 1] - diff[idx])\n    return r_t\n\n\n# Compute turning points for modes similar to those in Fig. 1\n# Paper examples: ell=0 v=3310, ell=20 v=3375, ell=70 v=3405 μHz\nmodes = [\n    (0, 3310, \"red\"),\n    (2, 3310, \"darkred\"),\n    (20, 3375, \"green\"),\n    (50, 3400, \"blue\"),\n    (70, 3405, \"purple\"),\n    (100, 3400, \"orange\"),\n    (200, 3400, \"brown\"),\n]\n\nprint(\"Mode turning points / 모드 전환점:\")\nprint(f\"{'ℓ':>5} {'ν (μHz)':>10} {'r_t/R☉':>10} {'Depth (R☉)':>12}\")\nprint(\"-\" * 42)\nfor ell, nu, _ in modes:\n    rt = find_turning_point(r_frac, c_profile, nu, ell)\n    depth = 1.0 - rt if not np.isnan(rt) else np.nan\n    print(f\"{ell:5d} {nu:10.0f} {rt:10.3f} {depth:12.3f}\")\n\n# Comprehensive plot: turning point depth vs ell for fixed frequency\nells = np.arange(0, 251, 1)\nnu_fixed = 3000.0  # μHz\nrt_arr = np.array([find_turning_point(r_frac, c_profile, nu_fixed, l) for l in ells])\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\nax = axes[0]\nax.plot(ells, rt_arr, \"b-\", lw=2)\nax.axhline(y=0.713, color=\"red\", ls=\"--\", alpha=0.7, label=\"CZ base (0.713)\")\nax.axhline(y=0.0, color=\"gray\", ls=\":\", alpha=0.5)\nax.set_xlabel(r\"Degree $\\ell$\")\nax.set_ylabel(r\"$r_t / R_\\odot$\")\nax.set_title(f\"Turning Point vs Degree (ν = {nu_fixed:.0f} μHz)\\n전환점 vs 차수\")\nax.legend()\nax.set_xlim(0, 250)\nax.set_ylim(0, 1)\n\n# Right: turning point vs frequency for fixed ell values\nax = axes[1]\nnus = np.linspace(1000, 5000, 200)\nfor ell_val, color, ls in [(0, \"red\", \"-\"), (20, \"green\", \"--\"), (100, \"blue\", \"-.\")]:\n    rts = [find_turning_point(r_frac, c_profile, nu, ell_val) for nu in nus]\n    ax.plot(nus, rts, color=color, ls=ls, lw=2, label=f\"ℓ = {ell_val}\")\nax.axhline(y=0.713, color=\"red\", ls=\":\", alpha=0.5, label=\"CZ base\")\nax.set_xlabel(\"Frequency ν [μHz]\")\nax.set_ylabel(r\"$r_t / R_\\odot$\")\nax.set_title(\"Turning Point vs Frequency / 전환점 vs 진동수\")\nax.legend()\nax.set_xlim(1000, 5000)\nax.set_ylim(0, 1)\n\nplt.tight_layout()\nplt.show()

## 4. Gamow Peak / Gamow 피크\n\nNuclear fusion in the solar core occurs via quantum tunneling through the Coulomb barrier. The reaction rate integrand is the product of:\n태양 핵에서의 핵융합은 Coulomb 장벽을 통한 양자 터널링으로 발생합니다. 반응률 피적분함수는:\n\n- **Maxwell-Boltzmann tail**: $\\exp(-E / k_B T)$ — fewer particles at higher energy\n  고에너지 입자가 적음\n- **Tunneling probability**: $\\exp(-\\sqrt{E_G / E})$ — higher energy ⟹ easier tunneling\n  고에너지일수록 터널링 확률 높음\n\nTheir product peaks at the **Gamow energy** $E_0$:\n이 둘의 곱은 **Gamow 에너지** $E_0$에서 피크를 형성합니다:\n\n$$E_0 = \\left(\\frac{E_G (k_B T)^2}{4}\\right)^{1/3}$$\n\nFor the pp reaction at $T_c \\approx 1.57 \\times 10^7$ K, the Gamow peak is at $E_0 \\approx 6$ keV.

In [ ]:
# Gamow Peak calculation for pp reaction\n\n# Constants in CGS / keV units\nalpha_fs = 1.0 / 137.036  # fine structure constant\nm_p_kg = 1.673e-27        # proton mass [kg]\nm_r = m_p_kg / 2           # reduced mass for pp [kg]\nhbar = 1.055e-34           # [J s]\ne_charge = 1.602e-19       # [C]\nk_B_eV = 8.617e-5          # Boltzmann constant [eV/K]\n\n# Gamow energy for pp reaction (Z1=Z2=1)\n# E_G = 2 * m_r * c^2 * (pi * alpha * Z1 * Z2)^2\n# Using E_G = (pi * alpha)^2 * 2 * m_r * c^2\nm_r_c2_keV = m_r * (3e8)**2 / (1.602e-16)  # in keV\nE_G_keV = 2 * m_r_c2_keV * (np.pi * alpha_fs * 1 * 1)**2\nprint(f\"Gamow energy E_G = {E_G_keV:.1f} keV\")\n\n# Core temperature\nT_core = 1.57e7  # K\nkT_keV = k_B_eV * T_core / 1000  # in keV\nprint(f\"k_B T_core = {kT_keV:.3f} keV\")\n\n# Gamow peak energy\nE0 = (E_G_keV * kT_keV**2 / 4)**(1.0 / 3.0)\nprint(f\"Gamow peak energy E_0 = {E0:.2f} keV\")\n\n# Width of the Gamow peak\nDelta = 4 * np.sqrt(E0 * kT_keV / 3)\nprint(f\"Gamow peak width Δ = {Delta:.2f} keV\")\n\n# Plot the three factors\nE = np.linspace(0.01, 30, 1000)  # keV\n\n# Maxwell-Boltzmann factor\nmb = np.exp(-E / kT_keV)\n\n# Tunneling factor\ntunnel = np.exp(-np.sqrt(E_G_keV / E))\n\n# Combined (Gamow integrand)\ngamow = np.exp(-E / kT_keV - np.sqrt(E_G_keV / E))\n\n# Normalize for visualization\nmb_norm = mb / np.max(mb)\ntunnel_norm = tunnel / np.max(tunnel)\ngamow_norm = gamow / np.max(gamow)\n\nfig, ax = plt.subplots(figsize=(10, 6))\n\nax.plot(E, mb_norm, \"b--\", lw=2, label=r\"Maxwell-Boltzmann: $e^{-E/k_BT}$\")\nax.plot(E, tunnel_norm, \"r--\", lw=2, label=r\"Tunneling: $e^{-\\sqrt{E_G/E}}$\")\nax.fill_between(E, gamow_norm, alpha=0.3, color=\"green\")\nax.plot(E, gamow_norm, \"g-\", lw=3, label=f\"Gamow peak (product)\")\n\n# Mark the peak\nax.axvline(E0, color=\"black\", ls=\":\", alpha=0.7)\nax.annotate(\n    f\"$E_0$ = {E0:.1f} keV\\nΔ = {Delta:.1f} keV\",\n    xy=(E0, 1.0), xytext=(E0 + 4, 0.8),\n    fontsize=12, arrowprops=dict(arrowstyle=\"->\", color=\"black\"),\n    bbox=dict(boxstyle=\"round,pad=0.3\", facecolor=\"lightyellow\"),\n)\n\n# Mark thermal energy\nax.axvline(kT_keV, color=\"blue\", ls=\":\", alpha=0.5)\nax.text(kT_keV + 0.2, 0.6, f\"$k_BT$ = {kT_keV:.2f} keV\",\n        color=\"blue\", fontsize=10)\n\nax.set_xlabel(\"Energy E [keV]\")\nax.set_ylabel(\"Normalized integrand\")\nax.set_title(\n    \"Gamow Peak for pp Reaction at Solar Core Temperature\\n\"\n    \"태양 핵 온도에서의 pp 반응 Gamow 피크\"\n)\nax.legend(loc=\"upper right\")\nax.set_xlim(0, 25)\nax.set_ylim(0, 1.15)\n\nplt.tight_layout()\nplt.show()\n\nprint(f\"\\n--- Summary / 요약 ---\")\nprint(f\"Coulomb barrier for pp: ~1 MeV (= {1000:.0f} keV)\")\nprint(f\"Thermal energy k_BT:   {kT_keV:.2f} keV (ratio: {1000/kT_keV:.0f}× too low)\")\nprint(f\"Gamow peak E_0:        {E0:.2f} keV\")\nprint(f\"→ Fusion occurs via quantum tunneling in the tail of Maxwell-Boltzmann distribution\")\nprint(f\"→ 핵융합은 Maxwell-Boltzmann 분포의 꼬리에서 양자 터널링으로 발생\")

## 5. EOS Comparison: Effect on γ₁ / 상태방정식 비교: γ₁에 미치는 효과\n\nThe paper (Fig. 3) shows how different equations of state affect $\\gamma_1$ and sound speed. The key physics is **ionization**: in partially ionized regions, energy goes into ionization rather than increasing temperature, reducing $\\gamma_1$ below 5/3.\n\n논문 Fig. 3은 서로 다른 상태방정식이 $\\gamma_1$과 음속에 미치는 영향을 보여줍니다. 핵심 물리는 **이온화**입니다: 부분 이온화 영역에서 에너지가 온도 증가 대신 이온화에 사용되어 $\\gamma_1$이 5/3 아래로 감소합니다.\n\nWe model three regions:\n세 영역을 모델링합니다:\n\n1. **Fully ionized interior**: $\\gamma_1 = 5/3$ (ideal gas)\n2. **Helium ionization zones**: $\\gamma_1$ dips (He I at ~40,000 K, He II at ~160,000 K)\n3. **Hydrogen ionization zone**: $\\gamma_1$ dip near surface (~10,000 K)

In [ ]:
def saha_ionization(T, n_e, chi, g_ratio=1.0):\n    \"\"\"Saha equation: ionization fraction x.\n\n    Args:\n        T: Temperature [K].\n        n_e: Electron number density [cm^-3].\n        chi: Ionization energy [eV].\n        g_ratio: Statistical weight ratio g_{i+1}/g_i.\n\n    Returns:\n        Ionization fraction x.\n    \"\"\"\n    chi_erg = chi * 1.602e-12  # eV to erg\n    kT = k_B * T\n    # Saha factor\n    thermal_lambda = np.sqrt(2 * np.pi * 9.109e-28 * kT) / 1.055e-27 / (2 * np.pi)\n    saha = (g_ratio / n_e) * (2 * np.pi * 9.109e-28 * kT / (2 * np.pi * 1.055e-27)**2)**1.5 * np.exp(-chi_erg / kT)\n    # x^2/(1-x) = saha => solve quadratic\n    # x = (-saha + sqrt(saha^2 + 4*saha)) / 2\n    x = (-saha + np.sqrt(saha**2 + 4 * saha)) / 2\n    return np.clip(x, 0, 1)\n\n\ndef gamma1_with_ionization(T, rho, X=0.7, Y=0.28):\n    \"\"\"Compute approximate gamma_1 including ionization effects.\n\n    Simplified model: gamma_1 drops in ionization zones.\n\n    Args:\n        T: Temperature array [K].\n        rho: Density array [g/cm^3].\n        X: Hydrogen mass fraction.\n        Y: Helium mass fraction.\n\n    Returns:\n        gamma_1 array.\n    \"\"\"\n    gamma1 = np.full_like(T, 5.0 / 3.0)\n\n    # Approximate ionization fractions using Gaussian dips\n    # H ionization: chi = 13.6 eV, T ~ 10,000 K\n    # He I ionization: chi = 24.6 eV, T ~ 40,000 K\n    # He II ionization: chi = 54.4 eV, T ~ 160,000 K\n\n    log_T = np.log10(T)\n\n    # H ionization dip (surface, log T ~ 4.0)\n    dip_H = 0.35 * X * np.exp(-((log_T - 4.0) / 0.15)**2)\n\n    # He I ionization dip (log T ~ 4.5)\n    dip_He1 = 0.15 * Y * np.exp(-((log_T - 4.55) / 0.12)**2)\n\n    # He II ionization dip (log T ~ 5.2) — this is the one probed by helioseismology\n    dip_He2 = 0.20 * Y * np.exp(-((log_T - 5.2) / 0.15)**2)\n\n    gamma1 -= (dip_H + dip_He1 + dip_He2)\n    return gamma1\n\n\n# Create temperature-density profile using polytrope\n# T goes from T_c=1.57e7 to T_surface~5800 K\nr_norm = xi3 / xi3[-1]\nT_poly = T_c * theta3\n# Ensure minimum temperature at surface\nT_poly = np.maximum(T_poly, 5800)\nrho_poly = rho_c * theta3**3\nrho_poly = np.maximum(rho_poly, 1e-7)\n\n# Compute gamma_1 profiles\ngamma1_ideal = np.full_like(T_poly, 5.0 / 3.0)  # EFF-like (no ionization effects)\ngamma1_mhd = gamma1_with_ionization(T_poly, rho_poly, X=0.7, Y=0.28)\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Left: gamma_1 profile\nax = axes[0]\nax.plot(r_norm, gamma1_ideal, \"b--\", lw=2, alpha=0.7, label=r\"Ideal gas: $\\gamma_1 = 5/3$\")\nax.plot(r_norm, gamma1_mhd, \"r-\", lw=2, label=r\"With ionization (MHD-like)\")\nax.axvline(x=0.713, color=\"gray\", ls=\":\", alpha=0.5, label=\"CZ base\")\nax.axhline(y=5/3, color=\"gray\", ls=\":\", alpha=0.3)\n\n# Annotate ionization zones\nax.annotate(\"He II\\nionization\", xy=(0.975, 1.52), fontsize=9,\n            ha=\"center\", color=\"darkred\")\nax.annotate(\"He I\", xy=(0.985, 1.58), fontsize=9,\n            ha=\"center\", color=\"darkred\")\n\nax.set_xlabel(r\"$r / R_\\odot$\")\nax.set_ylabel(r\"$\\gamma_1$\")\nax.set_title(r\"Adiabatic Exponent $\\gamma_1$ / 단열 지수\")\nax.legend(loc=\"lower left\")\nax.set_xlim(0, 1)\nax.set_ylim(1.1, 1.72)\n\n# Right: zoom into outer layers where ionization effects matter\nax = axes[1]\nouter = r_norm > 0.9\nax.plot(r_norm[outer], gamma1_ideal[outer], \"b--\", lw=2, alpha=0.7, label=\"Ideal gas\")\nax.plot(r_norm[outer], gamma1_mhd[outer], \"r-\", lw=2, label=\"With ionization\")\nax.axhline(y=5/3, color=\"gray\", ls=\":\", alpha=0.3)\n\n# Show the He II ionization zone (~15,000 km depth = r/R ~ 0.978)\nax.axvspan(0.97, 0.985, alpha=0.1, color=\"orange\", label=\"He II zone (~15,000 km)\")\n\nax.set_xlabel(r\"$r / R_\\odot$\")\nax.set_ylabel(r\"$\\gamma_1$\")\nax.set_title(r\"Outer layers: ionization zones / 외층: 이온화 영역\")\nax.legend(loc=\"lower left\", fontsize=10)\nax.set_xlim(0.9, 1.0)\nax.set_ylim(1.1, 1.72)\n\nplt.tight_layout()\nplt.show()\n\nprint(\"The He II ionization zone at ~15,000 km depth is used to:\")\nprint(\"  1. Test the equation of state (EOS)\")\nprint(\"  2. Determine the helium abundance Y☉ ≈ 0.24\")\nprint(\"He II 이온화 영역(~15,000 km 깊이)은 다음에 사용됩니다:\")\nprint(\"  1. 상태방정식(EOS) 시험\")\nprint(\"  2. 헬륨 존재비 Y☉ ≈ 0.24 결정\")

## 6. Neutrino Flux: Model Prediction vs Observation / 중성미자 유량: 모델 예측 vs 관측\n\nThe paper reports specific neutrino flux values for Model S vs. three experiments.\nModel S predicts substantially higher fluxes than observed — the Solar Neutrino Problem.\n\n논문은 Model S의 중성미자 예측값과 세 가지 실험 관측값을 구체적으로 보고합니다.\nModel S는 관측값보다 크게 높은 유량을 예측합니다 — 태양 중성미자 문제.

In [ ]:
# Neutrino flux comparison from the paper (Table in note 11)\n\nexperiments = [\"³⁷Cl\\n(Homestake)\", \"⁷¹Ga\\n(GALLEX/SAGE)\", \"⁸B\\n(Kamiokande)\"]\nunits = [\"SNU\", \"SNU\", r\"$\\times 10^6$ cm⁻²s⁻¹\"]\nobserved = [2.3, 78, 2.9]\npredicted = [8.2, 132, 5.9]\nratios = [o / p for o, p in zip(observed, predicted)]\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Left: bar chart comparison\nax = axes[0]\nx = np.arange(len(experiments))\nwidth = 0.35\nbars1 = ax.bar(x - width / 2, predicted, width, label=\"Model S prediction\",\n               color=\"steelblue\", alpha=0.8)\nbars2 = ax.bar(x + width / 2, observed, width, label=\"Observed\",\n               color=\"coral\", alpha=0.8)\n\nax.set_ylabel(\"Flux [various units]\")\nax.set_title(\"Solar Neutrino Problem: Model S vs Observation\\n태양 중성미자 문제\")\nax.set_xticks(x)\nax.set_xticklabels(experiments)\nax.legend()\n\n# Add ratio labels\nfor i, (pred, obs) in enumerate(zip(predicted, observed)):\n    ax.text(i, max(pred, obs) + 3, f\"Obs/Pred = {obs/pred:.2f}\",\n            ha=\"center\", fontsize=10, color=\"darkred\")\n\n# Right: the deficit as ratio (observed/predicted)\nax = axes[1]\ncolors = [\"steelblue\", \"green\", \"coral\"]\nbars = ax.bar(experiments, ratios, color=colors, alpha=0.8, edgecolor=\"black\")\nax.axhline(y=1.0, color=\"black\", ls=\"--\", lw=2, label=\"Perfect agreement\")\nax.axhline(y=1/3, color=\"red\", ls=\":\", alpha=0.7, label=\"1/3 (expected for 3-flavor oscillation)\")\nax.set_ylabel(\"Observed / Predicted\")\nax.set_title(\"Neutrino Deficit Ratio / 중성미자 결핍 비율\")\nax.set_ylim(0, 1.2)\nax.legend()\n\nfor bar, ratio in zip(bars, ratios):\n    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,\n            f\"{ratio:.2f}\", ha=\"center\", fontsize=12, fontweight=\"bold\")\n\nplt.tight_layout()\nplt.show()\n\nprint(\"\\nKey insight from the paper / 논문의 핵심 통찰:\")\nprint(\"Since helioseismology confirms the SSM to <0.5% in sound speed,\")\nprint(\"the neutrino deficit is unlikely to be caused by model errors.\")\nprint(\"→ The solution must lie in neutrino physics (oscillations).\")\nprint(\"\\n일진학이 SSM의 음속을 <0.5%로 확인했으므로,\")\nprint(\"중성미자 결핍은 모델 오류로 인한 것이 아닐 가능성이 높습니다.\")\nprint(\"→ 해결책은 중성미자 물리학(진동)에 있어야 합니다.\")\nprint(f\"\\nNote: c² ∝ T/μ — helioseismology constrains T/μ, not T alone.\")\nprint(f\"참고: c² ∝ T/μ — 일진학은 T/μ를 제약하지만, T를 단독으로 결정하지는 못합니다.\")

## Summary / 요약\n\nThis notebook implemented the following key concepts from Christensen-Dalsgaard et al. (1996):\n이 노트북은 다음 핵심 개념을 구현했습니다:\n\n| Section | Concept / 개념 | Key Result / 핵심 결과 |\n|---|---|---|\n| 1 | Lane-Emden polytrope | n=3 model reproduces basic solar density structure |\n| 2 | Sound speed profile | $c^2 = \\gamma_1 k_B T / \\mu m_p$ — central c ≈ 500 km/s |\n| 3 | p-mode turning points | $c(r_t)/r_t = 2\\pi\\nu/(\\ell+1/2)$ — low ℓ probes core |\n| 4 | Gamow peak | $E_0 \\approx 6$ keV for pp at $T_c = 1.57 \\times 10^7$ K |\n| 5 | EOS / γ₁ comparison | He II ionization zone dip used to determine Y☉ |\n| 6 | Neutrino problem | Obs/Pred ≈ 0.28–0.59 → neutrino oscillation solution |\n\n**The paper's central message**: improvements in microphysics (OPAL opacity, modern EOS, gravitational settling) brought the SSM into <0.5% agreement with helioseismology — confirming the model and pointing to neutrino physics as the source of the neutrino deficit.\n\n**논문의 핵심 메시지**: 미시물리학의 개선(OPAL opacity, 현대적 EOS, 중력 침강)이 SSM을 일진학과 <0.5% 이내로 일치시켰으며, 이는 모델의 정확성을 확인하고 중성미자 문제의 원인이 중성미자 물리학에 있음을 시사합니다.